# Do Models Fail Where Humans Disagree? — full pipeline

Methodological replication of **Nemani & Garimella (2026)**, *Large-Scale Multimodal Content Analysis and Annotation with Vision-Language Models* (ICWSM 2026), on the public **MMHS150K** dataset, plus a novel analysis of model errors vs. human annotator disagreement.

**Runtime:** Google Colab, free tier, T4 GPU. All outputs are written to Google Drive so every step can be resumed after a disconnect.

| Part | What it does | Hardware |
|---|---|---|
| 0 | Setup: Drive, Kaggle, folders | CPU/GPU |
| 1 | Download labels + splits, build binary labels and agreement, draw the stratified 3,000 sample | CPU |
| 2 | Extract images + OCR for the sample, build the master table | CPU |
| 3 | VLM image descriptions (Qwen2.5-VL-3B, prompt C.9), resumable | **GPU** |
| 4 | Zero-shot classification (Qwen2.5-7B, prompt C.3), 4 input settings, P(true) scoring | **GPU — restart runtime first** |
| 5 | Checks: free-generation agreement (200) + reasoning-mode robustness (240) | **GPU** |
| 6 | Trained baselines: LaBSE + logistic regression ± SMOTE | CPU |
| 7 | Replication evaluation: bootstrap CIs, McNemar, paired ΔF1 (Holm) | CPU |
| 8 | Novel analysis: agreement strata, Krippendorff's α substitution, soft labels | CPU |
| 9 | Figures, error analysis, slur check | CPU |

Seed: **42** everywhere.

## Part 0 — Setup

In [ ]:
!pip -q install -U bitsandbytes accelerate kaggle sentence-transformers imbalanced-learn krippendorff

from google.colab import drive
drive.mount('/content/drive')

import os, re, json, time, gc, glob, shutil, zipfile, subprocess
import numpy as np, pandas as pd, torch
from tqdm.auto import tqdm

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)

PROJ = '/content/drive/MyDrive/vlm_hate_replication'
DATA, OUT, FIG = f'{PROJ}/data', f'{PROJ}/outputs', f'{PROJ}/figures'
for d in ['data', 'outputs', 'prompts', 'figures', 'secrets']:
    os.makedirs(f'{PROJ}/{d}', exist_ok=True)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
# Kaggle API key: uploaded once, then kept in Drive/secrets (never commit this folder)
from google.colab import files
SECRET = f'{PROJ}/secrets/kaggle.json'
if not os.path.exists(SECRET):
    up = files.upload()
    shutil.move(list(up.keys())[0], SECRET)
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy(SECRET, '/root/.kaggle/kaggle.json'); os.chmod('/root/.kaggle/kaggle.json', 0o600)
DS = 'victorcallejasf/multimodal-hate-speech'

## Part 1 — Labels, agreement and the stratified sample

In [ ]:
RAW = '/content/raw'; os.makedirs(RAW, exist_ok=True)

def kget(path, dest=RAW):
    r = subprocess.run(['kaggle', 'datasets', 'download', DS, '-f', path, '-p', dest], capture_output=True, text=True)
    for z in glob.glob(f'{dest}/**/*.zip', recursive=True):          # Kaggle sometimes zips single files
        with zipfile.ZipFile(z) as zf: zf.extractall(os.path.dirname(z))
        os.remove(z)
    print('OK ' if r.returncode == 0 else 'FAIL', path)

for f in ['MMHS150K_GT.json', 'MMHS150K_readme.txt', 'hatespeech_keywords.txt',
          'splits/train_ids.txt', 'splits/val_ids.txt', 'splits/test_ids.txt']:
    kget(f)
for p in glob.glob(f'{RAW}/**/*.*', recursive=True):
    shutil.copy(p, DATA)

In [ ]:
gt = json.load(open(f'{DATA}/MMHS150K_GT.json'))
splits = {s: [l.strip() for l in open(f'{DATA}/{s}_ids.txt') if l.strip()] for s in ['train', 'val', 'test']}
print("GT:", len(gt), {s: len(v) for s, v in splits.items()})

rows, excluded = [], []
for tid in splits['test']:
    labs = gt[tid]['labels']
    if len(labs) != 3:                       # 2 test tweets do not have exactly 3 labels
        excluded.append(dict(id=tid, labels=labs)); continue
    h = sum(l != 0 for l in labs)            # number of 'hate' votes (any label 1-5)
    rows.append(dict(id=tid, tweet_text=gt[tid]['tweet_text'], l1=labs[0], l2=labs[1], l3=labs[2],
                     n_hate=h, hate_score=h / 3, label=int(h >= 2),
                     agreement='3/3' if h in (0, 3) else '2/3'))
test = pd.DataFrame(rows)
test.to_csv(f'{DATA}/test_all_labels.csv', index=False)
pd.DataFrame(excluded).to_csv(f'{DATA}/test_excluded_not3labels.csv', index=False)
print("kept", len(test), "| excluded", len(excluded))
print(pd.crosstab(test['label'], test['agreement'], margins=True))

In [ ]:
test['stratum'] = test['label'].map({0: 'nothate', 1: 'hate'}) + '_' + test['agreement']
sample = (test.groupby('stratum', group_keys=False)
              .apply(lambda g: g.sample(n=750, random_state=SEED))
              .sample(frac=1, random_state=SEED).reset_index(drop=True))
sample.to_csv(f'{DATA}/sample_3000.csv', index=False)
open(f'{DATA}/sample_3000_ids.txt', 'w').write('\n'.join(sample['id']))
print(pd.crosstab(sample['label'], sample['agreement'], margins=True))

## Part 2 — Images, OCR and the master table
The full dataset zip (~6.4 GB) is downloaded once to local disk; only the sample's images and OCR files are kept, backed up to Drive as one zip.

In [ ]:
WORK = '/content/work'; IMG_DIR, OCR_DIR = f'{WORK}/images', f'{WORK}/ocr'
os.makedirs(IMG_DIR, exist_ok=True); os.makedirs(OCR_DIR, exist_ok=True)
BACKUP = f'{DATA}/sample_media.zip'
ids = set(sample['id'])

if os.path.exists(BACKUP):
    with zipfile.ZipFile(BACKUP) as zf: zf.extractall(WORK)
else:
    !kaggle datasets download {DS} -p /content -q
    BIG = glob.glob('/content/*.zip')[0]
    with zipfile.ZipFile(BIG) as zf:
        for n in zf.namelist():
            stem, folder = os.path.splitext(os.path.basename(n))[0], n.split('/')[0]
            if stem in ids and folder in ('img_resized', 'img_txt'):
                dest = IMG_DIR if folder == 'img_resized' else OCR_DIR
                with zf.open(n) as src, open(f'{dest}/{os.path.basename(n)}', 'wb') as dst:
                    shutil.copyfileobj(src, dst)
    os.remove(BIG)
    shutil.make_archive(BACKUP[:-4], 'zip', WORK)
print("images:", len(os.listdir(IMG_DIR)), "| OCR files:", len(os.listdir(OCR_DIR)))

In [ ]:
def clean_tweet(t):
    return re.sub(r'\s+', ' ', re.sub(r'https?://t\.co/\S+', '', t)).strip()

def load_ocr(tid):
    p = f'{OCR_DIR}/{tid}.json'
    return re.sub(r'\s+', ' ', json.load(open(p)).get('img_text', '')).strip() if os.path.exists(p) else ''

master = sample.copy()
master['text_clean'] = master['tweet_text'].apply(clean_tweet)
master['ocr_text'] = master['id'].apply(load_ocr)
master['has_ocr'] = master['ocr_text'].str.len() > 0
master['img_path'] = master['id'].apply(lambda t: f'{IMG_DIR}/{t}.jpg')
master.to_csv(f'{DATA}/master_3000.csv', index=False)
print("has OCR:", master['has_ocr'].sum())

## Part 3 — VLM image descriptions (GPU)
Prompt = Nemani & Garimella App. C.9, with the modality set to *Image*, the audio clause removed and a JSON-only instruction added. The VLM sees the image only, never the tweet.

In [ ]:
from PIL import Image
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig

VLM_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
vlm = Qwen2_5_VLForConditionalGeneration.from_pretrained(VLM_ID, dtype=torch.float16, device_map='cuda',
                                                         attn_implementation='sdpa').eval()
proc = AutoProcessor.from_pretrained(VLM_ID, min_pixels=256*28*28, max_pixels=640*28*28)
proc.tokenizer.padding_side = 'left'

DESC_PROMPT = open('prompts/C9_image_description.txt').read() if os.path.exists('prompts/C9_image_description.txt') else '''Please provide a detailed description of this Image.
Instructions:
1. Analyze thoroughly including: visual content and scenes, text overlays or captions, overall narrative or message.
2. Provide comprehensive description capturing: main topic/subject matter, key points/arguments/story, tone and style, notable features/editing/production elements.
3. Keep description clear and concise (3-4 sentences).
Output Schema:
{"type": "object", "properties": {"description": {"type": "string"}}, "required": ["description"]}
Respond only with JSON matching the schema.'''

def describe(paths, max_new_tokens=200):
    msgs = [[{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": DESC_PROMPT}]}] for _ in paths]
    texts = [proc.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in msgs]
    inputs = proc(text=texts, images=[Image.open(p).convert('RGB') for p in paths], padding=True, return_tensors='pt').to('cuda')
    with torch.inference_mode():
        out = vlm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return proc.batch_decode(out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)

def parse_desc(raw):
    s = raw.strip().removeprefix('```json').removeprefix('```').removesuffix('```').strip()
    try:    return json.loads(s)['description'], True
    except Exception: return s, False

In [ ]:
# resumable run: appends to JSONL on Drive, skips IDs already done
OUT_DESC = f'{OUT}/descriptions_qwen25vl3b.jsonl'
done = set()
if os.path.exists(OUT_DESC):
    for line in open(OUT_DESC):
        try: done.add(json.loads(line)['id'])
        except Exception: pass
todo = master[~master['id'].isin(done)].reset_index(drop=True)
BATCH_VLM = 8
with open(OUT_DESC, 'a') as f:
    for i in tqdm(range(0, len(todo), BATCH_VLM)):
        chunk = todo.iloc[i:i + BATCH_VLM]
        try:
            raws = describe(chunk['img_path'].tolist())
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); raws = [describe([p])[0] for p in chunk['img_path']]
        for tid, raw in zip(chunk['id'], raws):
            d, ok = parse_desc(raw)
            f.write(json.dumps(dict(id=tid, description=d, json_ok=ok, raw=raw, model=VLM_ID)) + '\n')
        f.flush(); os.fsync(f.fileno())

In [ ]:
# repair: re-run unparsed outputs one at a time (400 tokens); salvage text; flag fp16-degenerate outputs
desc = pd.DataFrame([json.loads(l) for l in open(OUT_DESC) if l.strip()]).drop_duplicates('id', keep='first')
for i in desc.index[~desc['json_ok']]:
    raw = describe([f"{IMG_DIR}/{desc.at[i, 'id']}.jpg"], max_new_tokens=400)[0]
    d, ok = parse_desc(raw)
    desc.loc[i, ['description', 'json_ok', 'raw']] = [d, ok, raw]

def salvage(raw):
    s = raw.strip().removeprefix('```json').removeprefix('```').strip()
    m = re.search(r'"description"\s*:\s*"(.*)', s, flags=re.S)
    if m: s = m.group(1)
    s = s.replace('\\"', '"').rstrip('"}` \n')
    sents = re.split(r'(?<=[.!?])\s+', s)
    if len(sents) > 1 and not re.search(r'[.!?]$', sents[-1]): sents = sents[:-1]
    return ' '.join(sents[:4]).strip()

desc['desc_status'] = np.where(desc['json_ok'], 'ok', 'salvaged')
for i in desc.index[~desc['json_ok']]:
    if desc.at[i, 'raw'].count('!') > 20:
        desc.loc[i, ['description', 'desc_status']] = ['', 'unavailable']
    else:
        desc.at[i, 'description'] = salvage(desc.at[i, 'raw'])
desc.to_json(f'{OUT}/descriptions_qwen25vl3b_final.jsonl', orient='records', lines=True)
m2 = master.merge(desc[['id', 'description', 'desc_status']], on='id', how='left')
m2.to_csv(f'{DATA}/master_with_desc.csv', index=False)
print(desc['desc_status'].value_counts())

## Part 4 — Zero-shot classification (GPU)
**Restart the runtime before this part** (Runtime → Restart session), then re-run Part 0 (setup cell only). Loading the 7B model while the VLM is still in memory runs out of GPU memory.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
m2 = pd.read_csv(f'{DATA}/master_with_desc.csv', dtype={'id': str}, keep_default_na=False)
m2['has_ocr'] = m2['has_ocr'].astype(str) == 'True'
for c in ['label', 'n_hate']: m2[c] = m2[c].astype(int)
m2['hate_score'] = m2['hate_score'].astype(float)

LLM_ID = 'Qwen/Qwen2.5-7B-Instruct'
tok = AutoTokenizer.from_pretrained(LLM_ID); tok.padding_side = 'left'
llm = AutoModelForCausalLM.from_pretrained(LLM_ID, device_map='cuda', dtype=torch.float16,
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                               bnb_4bit_compute_dtype=torch.float16)).eval()
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.1f} GB  (expect ~5.9)")

In [ ]:
HATE_TEMPLATE = '''Task: Determine whether this {MODALITY} contains hateful elements.
  Definition: Content that promotes violence or incites hatred against individuals/groups based on: race, ethnic origin, religion, disability, gender, age, veteran status, or sexual orientation/gender identity.
  Hateful content includes: dehumanizing speech or imagery, allegations of serious immorality/criminality, slurs or harmful stereotypes, serious insults/contempt/disgust, calls for exclusion or segregation.
  Instructions:
1. Analyze completely
2. Is this hateful content? (Yes/No)
Output Schema:
{{"type": "object", "properties": {{"hateful content": {{"type": "boolean"}}}}, "required": ["hateful content"]}}

{CONTENT}'''

def build_prompt(r, setting):
    desc = r['description'] if r['desc_status'] != 'unavailable' else '(unavailable)'
    if setting == 'd' and not r['has_ocr']: setting = 'a'          # no OCR -> identical to text-only
    mod, con = {'a': ('Text', f"Tweet text: {r['text_clean']}"),
                'b': ('Image', f"Image description: {desc}"),
                'c': ('Text and Image', f"Tweet text: {r['text_clean']}\nImage description: {desc}"),
                'd': ('Text and Image', f"Tweet text: {r['text_clean']}\nText in image (OCR): {r['ocr_text']}")}[setting]
    return HATE_TEMPLATE.format(MODALITY=mod, CONTENT=con)

def chat(p):
    return tok.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True)

PREFIX = '{"hateful content":'
T_IDS = sorted({tok.encode(s, add_special_tokens=False)[0] for s in [' true', 'true']})
F_IDS = sorted({tok.encode(s, add_special_tokens=False)[0] for s in [' false', 'false']})

@torch.inference_mode()
def score_prob(prompts):
    enc = tok([chat(p) + PREFIX for p in prompts], return_tensors='pt', padding=True).to('cuda')
    logp = torch.log_softmax(llm(**enc).logits[:, -1, :].float(), dim=-1)
    lt, lf = torch.logsumexp(logp[:, T_IDS], -1), torch.logsumexp(logp[:, F_IDS], -1)
    return torch.sigmoid(lt - lf).cpu().numpy(), (lt.exp() + lf.exp()).cpu().numpy()

@torch.inference_mode()
def score_free(prompts, max_new_tokens=20):
    enc = tok([chat(p) for p in prompts], return_tensors='pt', padding=True).to('cuda')
    out = llm.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False)
    raws = tok.batch_decode(out[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
    return [(None if (m := re.search(r'"hateful content"\s*:\s*(true|false)', r)) is None else int(m.group(1) == 'true'))
            for r in raws], raws

In [ ]:
BATCH = 16
def load_done(out):
    if not os.path.exists(out): return pd.DataFrame(columns=['id', 'setting', 'p_yes', 'mass'])
    d = pd.read_csv(out, dtype={'id': str}, on_bad_lines='skip')
    for c in ['p_yes', 'mass']: d[c] = pd.to_numeric(d[c], errors='coerce')
    return d.dropna(subset=['p_yes', 'mass']).drop_duplicates('id')

def run_setting(s):
    out = f'{OUT}/zs_{s}.csv'
    rows = m2[m2['has_ocr']] if s == 'd' else m2
    todo = rows[~rows['id'].isin(set(load_done(out)['id']))].reset_index(drop=True)
    if len(todo) == 0: return
    prompts = [build_prompt(r, s) for _, r in todo.iterrows()]
    order = np.argsort([len(p) for p in prompts], kind='stable')      # less padding per batch
    for i in tqdm(range(0, len(order), BATCH), desc=f'setting {s}'):
        idx = order[i:i + BATCH]
        p, mass = score_prob([prompts[j] for j in idx])
        pd.DataFrame(dict(id=todo['id'].values[idx], setting=s, p_yes=p, mass=mass)).to_csv(
            out, mode='a', header=not os.path.exists(out), index=False)

for s in 'abcd':
    run_setting(s)

res = {s: load_done(f'{OUT}/zs_{s}.csv')[['id', 'p_yes']] for s in 'abcd'}
res['d'] = pd.concat([res['d'], res['a'][~res['a']['id'].isin(res['d']['id'])]])   # no-OCR items = copy of (a)
zs = m2[['id', 'label', 'agreement', 'stratum', 'hate_score', 'has_ocr']].copy()
for s in 'abcd':
    zs = zs.merge(res[s].rename(columns={'p_yes': f'p_{s}'}), on='id')
    zs[f'pred_{s}'] = (zs[f'p_{s}'] > 0.5).astype(int)
zs.to_csv(f'{OUT}/zs_all_settings.csv', index=False)

## Part 5 — Decoding checks (GPU)

In [ ]:
m2i = m2.set_index('id')
# (i) 200-item agreement: free generation vs P(true) > 0.5
chk = zs.sample(200, random_state=SEED).reset_index(drop=True)
chk['setting'] = np.tile(list('abcd'), 50)
prompts = [build_prompt(m2i.loc[r['id']].to_dict(), r['setting']) for _, r in chk.iterrows()]
fp, fr = [], []
for i in range(0, 200, BATCH):
    a, b = score_free(prompts[i:i + BATCH], max_new_tokens=80); fp += a; fr += b
chk['free_pred'], chk['free_raw'] = fp, fr
un = chk.index[chk['free_pred'].isna()]
for i in range(0, len(un), BATCH):                                   # reasoned first -> give room to finish
    ids_ = un[i:i + BATCH]
    a, b = score_free([prompts[j] for j in ids_], max_new_tokens=300)
    chk.loc[ids_, 'free_pred_long'], chk.loc[ids_, 'free_raw_long'] = a, b
chk['prob_pred'] = [r[f'pred_{r["setting"]}'] for _, r in chk.iterrows()]
chk.to_csv(f'{OUT}/agreement_check_200.csv', index=False)

# (ii) reasoning-mode robustness: 240 OCR items (60 per stratum) x settings a, c, d
OUTR = f'{OUT}/reasoning_check.csv'
rc_items = m2[m2['has_ocr']].groupby('stratum', group_keys=False).apply(lambda g: g.sample(60, random_state=SEED), include_groups=False)
rc_ids = m2[m2['has_ocr']].loc[rc_items.index, 'id'].tolist()
jobs = [(i, s) for s in 'acd' for i in rc_ids]
if os.path.exists(OUTR):
    prev = pd.read_csv(OUTR, dtype={'id': str}, on_bad_lines='skip'); jobs = [j for j in jobs if j not in set(zip(prev['id'], prev['setting']))]
prompts = [build_prompt(m2i.loc[i].to_dict(), s) for i, s in jobs]
order = np.argsort([len(p) for p in prompts], kind='stable')
for k in tqdm(range(0, len(order), BATCH)):
    idx = order[k:k + BATCH]
    a, b = score_free([prompts[j] for j in idx], max_new_tokens=300)
    pd.DataFrame(dict(id=[jobs[j][0] for j in idx], setting=[jobs[j][1] for j in idx], free_pred=a, free_raw=b)).to_csv(
        OUTR, mode='a', header=not os.path.exists(OUTR), index=False)

## Part 6 — Trained baselines (CPU)

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE

m2 = pd.read_csv(f'{DATA}/master_with_desc.csv', dtype={'id': str}, keep_default_na=False)
m2['label'] = m2['label'].astype(int)
gt = json.load(open(f'{DATA}/MMHS150K_GT.json'))
train_ids = [l.strip() for l in open(f'{DATA}/train_ids.txt') if l.strip()]
tr = pd.DataFrame([dict(id=t, text_clean=clean_tweet(gt[t]['tweet_text']), label=int(sum(l != 0 for l in gt[t]['labels']) >= 2))
                   for t in train_ids if len(gt[t]['labels']) == 3])
assert len(set(tr['id']) & set(m2['id'])) == 0
tr20 = tr.sample(20000, random_state=SEED).reset_index(drop=True)
tr20.to_csv(f'{DATA}/baseline_train_20k.csv', index=False)

EMB = f'{OUT}/emb'; os.makedirs(EMB, exist_ok=True)
labse = SentenceTransformer('sentence-transformers/LaBSE', device='cpu')
def embed_resumable(texts, name, chunk=2000):
    for i in tqdm(range(0, len(texts), chunk), desc=name):
        f = f'{EMB}/{name}_{i:06d}.npy'
        if not os.path.exists(f): np.save(f, labse.encode(texts[i:i + chunk], batch_size=64, normalize_embeddings=True))
    return np.vstack([np.load(f) for f in sorted(glob.glob(f'{EMB}/{name}_*.npy'))])
X_test = embed_resumable(m2['text_clean'].tolist(), 'test3000_text')
X_train = embed_resumable(tr20['text_clean'].tolist(), 'train20k_text')

def fit_predict(X, y):
    clf = LogisticRegression(max_iter=2000, C=1.0, random_state=SEED).fit(X, y)
    p = clf.predict_proba(X_test)[:, 1]; return p, (p > 0.5).astype(int)
p_lr, pred_lr = fit_predict(X_train, tr20['label'].values)
X_sm, y_sm = SMOTE(random_state=SEED, k_neighbors=5).fit_resample(X_train, tr20['label'].values)
p_sm, pred_sm = fit_predict(X_sm, y_sm)
pd.DataFrame(dict(id=m2['id'], p_lr=p_lr, pred_lr=pred_lr, p_smote=p_sm, pred_smote=pred_sm)).to_csv(f'{OUT}/baseline_preds.csv', index=False)

## Part 7 — Replication evaluation (CPU)

In [ ]:
from sklearn.metrics import roc_auc_score
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

zs = pd.read_csv(f'{OUT}/zs_all_settings.csv', dtype={'id': str})
T = zs.merge(pd.read_csv(f'{OUT}/baseline_preds.csv', dtype={'id': str}), on='id'); assert len(T) == 3000
SYS = {'(a) Text only': ('pred_a', 'p_a'), '(b) Description only': ('pred_b', 'p_b'),
       '(c) Text + description': ('pred_c', 'p_c'), '(d) Text + OCR': ('pred_d', 'p_d'),
       'LaBSE + LR': ('pred_lr', 'p_lr'), 'LaBSE + LR + SMOTE': ('pred_smote', 'p_smote')}
B = 2000; rng = np.random.default_rng(SEED)

def boot_prf(y, yhat, idx):
    yb, pb = y[idx], yhat[idx]
    tp = ((yb == 1) & (pb == 1)).sum(1); fp = ((yb == 0) & (pb == 1)).sum(1); fn = ((yb == 1) & (pb == 0)).sum(1)
    return tp / np.maximum(tp + fp, 1), tp / np.maximum(tp + fn, 1), 2 * tp / np.maximum(2 * tp + fp + fn, 1)
def point_prf(y, yhat):
    tp = ((y == 1) & (yhat == 1)).sum(); fp = ((y == 0) & (yhat == 1)).sum(); fn = ((y == 1) & (yhat == 0)).sum()
    return tp / max(tp + fp, 1), tp / max(tp + fn, 1), 2 * tp / max(2 * tp + fp + fn, 1)
ci = lambda v: f"[{np.percentile(v, 2.5):.3f}, {np.percentile(v, 97.5):.3f}]"

def make_table(df, systems, idx):
    yy, rows = df['label'].values, []
    for name, (pc, prc) in systems.items():
        yhat = df[pc].values.astype(int); P, R, F = point_prf(yy, yhat); bP, bR, bF = boot_prf(yy, yhat, idx)
        rows.append({'System': name, 'P': round(P, 3), 'P 95% CI': ci(bP), 'R': round(R, 3), 'R 95% CI': ci(bR),
                     'F1': round(F, 3), 'F1 95% CI': ci(bF), 'AUROC': round(roc_auc_score(yy, df[prc]), 3) if prc else None})
    return pd.DataFrame(rows)

def mcnemar_tests(df, pairs, names):
    out = []
    for x, z in pairs:
        cx = df[names[x]].values.astype(int) == df['label'].values; cz = df[names[z]].values.astype(int) == df['label'].values
        tab = [[np.sum(cx & cz), np.sum(cx & ~cz)], [np.sum(~cx & cz), np.sum(~cx & ~cz)]]
        r = mcnemar(tab, exact=False, correction=True)
        out.append({'A': x, 'B': z, 'A right / B wrong': tab[0][1], 'A wrong / B right': tab[1][0], 'chi2': round(r.statistic, 2), 'p': r.pvalue})
    out = pd.DataFrame(out); out['p (Holm)'] = multipletests(out['p'], method='holm')[1]; return out

idx = rng.integers(0, len(T), size=(B, len(T)))
tab_main = make_table(T, SYS, idx); tab_main.to_csv(f'{OUT}/table_replication_main.csv', index=False)
names = {k: v[0] for k, v in SYS.items()}
pairs = [('(a) Text only', '(c) Text + description'), ('(a) Text only', '(d) Text + OCR'), ('(a) Text only', '(b) Description only'),
         ('(c) Text + description', '(d) Text + OCR'), ('LaBSE + LR + SMOTE', '(a) Text only'), ('LaBSE + LR + SMOTE', 'LaBSE + LR')]
mcnemar_tests(T, pairs, names).to_csv(f'{OUT}/mcnemar_main.csv', index=False)

rc = pd.read_csv(f'{OUT}/reasoning_check.csv', dtype={'id': str}, on_bad_lines='skip').drop_duplicates(['id', 'setting'])
rc['free_pred'] = pd.to_numeric(rc['free_pred'], errors='coerce').fillna(0).astype(int)        # unparsed -> not hate
W = rc.pivot(index='id', columns='setting', values='free_pred').add_prefix('free_').reset_index().merge(T, on='id')
SYS_R = {f'{m} ({s})': (f'{"pred" if m == "direct" else "free"}_{s}', f'p_{s}' if m == 'direct' else None) for s in 'acd' for m in ['direct', 'reasoning']}
idx_r = rng.integers(0, len(W), size=(B, len(W)))
make_table(W, SYS_R, idx_r).to_csv(f'{OUT}/table_reasoning_check.csv', index=False)
names_r = {k: v[0] for k, v in SYS_R.items()}

def delta_f1(df, A, Bc, idx):
    yy = df['label'].values; d = boot_prf(yy, df[A].values.astype(int), idx)[2] - boot_prf(yy, df[Bc].values.astype(int), idx)[2]
    dF = point_prf(yy, df[A].values.astype(int))[2] - point_prf(yy, df[Bc].values.astype(int))[2]
    return round(dF, 3), ci(d), min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))
rows = [dict(A=a, B=b, dF1=x[0], CI=x[1], p_boot=x[2], set='main') for a, b in pairs for x in [delta_f1(T, names[a], names[b], idx)]]
rows += [dict(A=a, B=b, dF1=x[0], CI=x[1], p_boot=x[2], set='reasoning') for a, b in
         [('reasoning (a)', 'reasoning (c)'), ('reasoning (a)', 'reasoning (d)'), ('reasoning (a)', 'direct (a)'), ('reasoning (c)', 'direct (c)')]
         for x in [delta_f1(W, names_r[a], names_r[b], idx_r)]]
dtab = pd.DataFrame(rows)
for s in dtab['set'].unique():
    m = dtab['set'] == s; dtab.loc[m, 'p_boot (Holm)'] = multipletests(dtab.loc[m, 'p_boot'], method='holm')[1]
dtab.to_csv(f'{OUT}/delta_f1_bootstrap.csv', index=False)
print(tab_main.to_string(index=False))

## Part 8 — Novel analysis: do models fail where humans disagree? (CPU)

In [ ]:
from scipy.stats import spearmanr
import krippendorff
from sklearn.metrics import cohen_kappa_score

# (i) performance by human agreement
rng2 = np.random.default_rng(SEED + 1)
sub = {g: T[T['agreement'] == g].reset_index(drop=True) for g in ['3/3', '2/3']}
idx_g = {g: rng2.integers(0, len(sub[g]), size=(B, len(sub[g]))) for g in sub}
rows = []
for name, (pc, prc) in SYS.items():
    r, bf = {'System': name}, {}
    for g in ['3/3', '2/3']:
        yy, yh = sub[g]['label'].values, sub[g][pc].values.astype(int); bf[g] = boot_prf(yy, yh, idx_g[g])[2]
        r[f'F1 {g}'] = round(point_prf(yy, yh)[2], 3); r[f'F1 {g} CI'] = ci(bf[g]); r[f'AUROC {g}'] = round(roc_auc_score(yy, sub[g][prc]), 3)
    r['dF1 (3/3-2/3)'] = round(r['F1 3/3'] - r['F1 2/3'], 3); r['dF1 CI'] = ci(bf['3/3'] - bf['2/3']); rows.append(r)
pd.DataFrame(rows).to_csv(f'{OUT}/table_by_agreement.csv', index=False)
pd.DataFrame({n: T.groupby('stratum').apply(lambda g: (g[pc].astype(int) == g['label']).mean(), include_groups=False)
              for n, (pc, _) in SYS.items()}).T.round(3).to_csv(f'{OUT}/table_accuracy_by_stratum.csv')

# (ii) model as a 4th annotator: Krippendorff's alpha substitution test
def kalpha(M):
    N, m = M.shape; n1 = M.sum(1); Do = (2 * n1 * (m - n1)).sum() / (N * m * (m - 1))
    p = M.mean(); return 1 - Do / (2 * p * (1 - p) * (N * m) / (N * m - 1))
T2 = T.merge(m2[['id', 'l1', 'l2', 'l3']], on='id')
H = (T2[['l1', 'l2', 'l3']].astype(int).values != 0).astype(int)
assert abs(kalpha(H) - krippendorff.alpha(reliability_data=H.T, level_of_measurement='nominal')) < 1e-6
rng3 = np.random.default_rng(SEED + 2); idx_k = rng3.integers(0, len(T2), size=(B, len(T2)))
subst = lambda H, mv: np.mean([kalpha(np.column_stack([np.delete(H, k, axis=1), mv])) for k in range(3)])
a_h = kalpha(H); kappa_hh = np.mean([cohen_kappa_score(H[:, i], H[:, j]) for i, j in [(0, 1), (0, 2), (1, 2)]])
rows = [{'System': 'Humans only (3 coders)', 'alpha': round(a_h, 3), 'alpha CI': ci(np.array([kalpha(H[i]) for i in idx_k])), 'kappa_hh': round(kappa_hh, 3)}]
for name, (pc, _) in SYS.items():
    mv = T2[pc].values.astype(int)
    d_boot = np.array([subst(H[i], mv[i]) - kalpha(H[i]) for i in idx_k[:500]])
    rows.append({'System': name, 'alpha': round(subst(H, mv), 3), 'dalpha': round(subst(H, mv) - a_h, 3), 'dalpha CI': ci(d_boot),
                 'kappa model-human': round(np.mean([cohen_kappa_score(mv, H[:, k]) for k in range(3)]), 3)})
pd.DataFrame(rows).to_csv(f'{OUT}/table_annotator_agreement.csv', index=False)

# (iii) soft labels
T['n_hate'] = (T['hate_score'] * 3).round().astype(int)
hs = T['hate_score'].values; rng4 = np.random.default_rng(SEED + 3); idx_s = rng4.integers(0, len(T), size=(B, len(T)))
brier_ref = np.mean((hs - hs.mean()) ** 2); rows, lvl = [], {}
for name, (_, prc) in SYS.items():
    p = T[prc].values; br = np.mean((p - hs) ** 2)
    rows.append({'System': name, 'Spearman rho': round(spearmanr(p, hs).statistic, 3),
                 'rho CI': ci(np.array([spearmanr(p[i], hs[i]).statistic for i in idx_s[:1000]])),
                 'Brier': round(br, 3), 'Brier skill': round(1 - br / brier_ref, 3),
                 'extreme share': round(((p < 0.05) | (p > 0.95)).mean(), 3)})
    lvl[name] = T.groupby('n_hate')[prc].mean().round(3)
pd.DataFrame(rows).to_csv(f'{OUT}/table_soft_labels.csv', index=False)
pd.DataFrame(lvl).T.to_csv(f'{OUT}/table_mean_p_by_votes.csv')

## Part 9 — Error analysis and slur check (CPU)
Error categories were assigned manually (see paper, Table 8). Slurs appear here only as search patterns.

In [ ]:
E = T.merge(m2[['id', 'text_clean', 'l1', 'l2', 'l3']], on='id')
wrong = E[E['pred_a'] != E['label']]
quota = {'hate_3/3': 8, 'hate_2/3': 7, 'nothate_3/3': 8, 'nothate_2/3': 7}
err30 = pd.concat([wrong[wrong['stratum'] == s].sample(n, random_state=SEED) for s, n in quota.items()])
err30[['id', 'stratum', 'p_a', 'p_c']].to_csv(f'{OUT}/error_analysis_30_ids.csv', index=False)   # IDs only

SLURS = {'n-word': r'\bnigg', 'dyke': r'\bdykes?\b', 'f-slur': r'\bfag', 'ableist': r'\bretard', 'gendered': r'\b(?:cunt|twat)'}
rows = []
for name, pat in list(SLURS.items()) + [('none of these', None)]:
    g = E[~E['text_clean'].str.contains('|'.join(SLURS.values()), case=False, regex=True)] if pat is None \
        else E[E['text_clean'].str.contains(pat, case=False, regex=True)]
    rows.append({'term': name, 'n': len(g), 'humans: hate': round(g['label'].mean(), 3),
                 'model (a): hate': round(g['pred_a'].mean(), 3), 'model accuracy': round((g['pred_a'] == g['label']).mean(), 3)})
pd.DataFrame(rows).to_csv(f'{OUT}/table_slur_analysis.csv', index=False)
print(pd.DataFrame(rows).to_string(index=False))